# Assignment

## Brief

Write the Python codes for the following questions.

## Instructions

Paste the answer as Python in the answer code section below each question.

### Question 1

Question: Implement a simple Thrift server and client that defines a `Student` struct with fields `name` (string), `age` (integer), and `courses` (list of strings). Include a service `School` with a method `enrollCourse` that takes a `Student` record and a course name, adds the course to the student's course list, and returns the updated `Student` record.

Answer:

**Thrift schema (student.thrift)**

In [4]:
%%writefile ../schema/student.thrift

struct Student {
  1: required string name,
  2: optional i64 age,
  3: optional list<string> course,
}

service School {
    Student enrollCourse(1: required Student student,2: required string course);
}


Writing ../schema/student.thrift


**Thrift server (student_server.py)**

In [5]:
%%writefile ../student_thrift_server.py
import thriftpy2
student_thrift = thriftpy2.load("./schema/student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_server

class School(object):
    def enrollCourse(self, student, course):
        student.course.append(course)
        return student

server = make_server(student_thrift.School, School(), client_timeout=None)
print("Chef is in the kitchen (Server started)...")
print("Press Ctrl+C to stop the server.")
server.serve()

Writing ../student_thrift_server.py


Run the thrift server

**Thrift client (student_client.py)**

In [6]:
# Enter your student_client code here
import thriftpy2
student_thrift = thriftpy2.load("../schema/student.thrift", module_name="student_thrift")

from thriftpy2.rpc import make_client

# Connect to the kitchen
school = make_client(student_thrift.School, timeout=None)

# Create a student named Martin
martin = student_thrift.Student(
    name="Martin", course=["math", "art"]
)
print(f"Initial courses: {martin.course}")

# Send Martin to school to learn 'coding' remotely!
martin = school.enrollCourse(martin, "biology")

# Let's see if he learned it
print(f"Updated courses: {martin.course}")

Initial courses: ['math', 'art']
Updated courses: ['math', 'art', 'biology']


### Question 2

Question: Implement a simple Protocol Buffers server and client that defines a `Book` message with fields `title` (string), `author` (string), and `page_count` (integer). Include a service `Library` with a method `checkoutBook` that takes a `Book` message and returns the same `Book` message.

Answer:

**Protobuf schema (book.proto)**

In [17]:
%%writefile ../schema/book.proto
syntax = "proto3";

message Book {
  string title = 1;
  string author = 2;
  optional int32 page_count = 3;
}
  
service Library {
  rpc checkoutBook(Book) returns (Book) {}
}

Overwriting ../schema/book.proto


Run

**Protobuf server (book_server.py)**

In [18]:
%%writefile ../book_server.py
from concurrent import futures
import grpc
import book_pb2
import book_pb2_grpc


class Library(book_pb2_grpc.LibraryServicer):

    def checkoutBook(self, request, context):
        # Validate page_count is a positive number
        if request.page_count < 0:
            context.abort(
                grpc.StatusCode.INVALID_ARGUMENT,
                "Page count cannot be negative"
            )

        print(f"Checking out: '{request.title}' by {request.author}")

        # Return the same Book back to the client
        return request


server = grpc.server(futures.ThreadPoolExecutor(max_workers=2))
book_pb2_grpc.add_LibraryServicer_to_server(Library(), server)
server.add_insecure_port('[::]:50051')
print("Library is open (Server started)...")
print("Press Ctrl+C to close the library.")
server.start()
server.wait_for_termination()

Overwriting ../book_server.py


Run

**Protobuf client (book_client.py)**

In [19]:
# Enter your book_client code here
import sys
sys.path.append('..')
import grpc
import book_pb2
import book_pb2_grpc


def checkout_book(stub, title, author, page_count):
    # Validate page_count before sending
    if page_count < 0:
        raise ValueError("Page count cannot be negative")

    # Create a Book message
    book = book_pb2.Book(
        title=title,
        author=author,
        page_count=page_count
    )

    print(f"Sending book to library: '{book.title}'")

    # Send to server via RPC call
    checked_out_book = stub.checkoutBook(book)
    return checked_out_book


# Connect to the library server
with grpc.insecure_channel('localhost:50051') as channel:
    stub = book_pb2_grpc.LibraryStub(channel)

    # Checkout a book
    book = checkout_book(
        stub,
        title="The Pragmatic Programmer",
        author="Andrew Hunt",
        page_count=352
    )

    print(f"\nSuccessfully checked out:")
    print(f"  Title      : {book.title}")
    print(f"  Author     : {book.author}")
    print(f"  Page Count : {book.page_count}")

Sending book to library: 'The Pragmatic Programmer'

Successfully checked out:
  Title      : The Pragmatic Programmer
  Author     : Andrew Hunt
  Page Count : 352
